# FIT3182 - Big data management and processing

# Activity: Advanced Operations in MongoDB #

In the previous tutorial, we learned fundamentals of MongoDB including its CRUD operations. This week, we will learn how we can create more complex queries to manipulate documents in a collection. Next week, we will learn more deeply about how to use CRUD operations in MongoDB in Python.

**In this activity, we will learn the following advanced features in MongoDB:**
- Advanced query operators and regular expressions. 
- MongoDB text search engine mechanism.
- Aggregation that is a great way to massage data into the output (including join two collections) 
- Designing data models for an application: `embedding` and `referencing`. 

Let's get started!

## Sample Collection
Let's first start two containers from the last lab: mongodb server and mongoshell. We will use the database, `fit3182_db` and the collection, `FIT_COMPLEX`.

Note that we will use the **`FIT_COMPLEX`** collection that we created by importing a json data in the last tutorial. If you have not created it yet, you need to look at the previous MongoDB tutorial material and need to create this collection.

For this week's activity, we will add more documents in the collection.
<font color='blue'>
**Exercise**: Add the following 3 students into the collection, "FIT_COMPLEX", using 'insert()'
```
{
    "sid": 125,
    "name": {
      "first": "John",
      "last": "Moody"
    },
    "course": "MDS",
    "result": [
      {
        "unit_code": "FIT5145",
        "unit_name": "Intoduction to Data Science",
        "synopsis":"This unit looks at processes, case studies and simple tools to understand the many facets of working with data, and the significant effort in Data Science over and above the core task of Data Analysis.",
        "semester": 2,
        "year": [2017],
        "mark": 90
      }
    ]
}

{
    "sid": 126,
    "name": {
      "first": "David",
      "last": "Lee"
    },
    "course": "MIT",
    "result": [
      {
        "unit_code": "FIT5136",
        "unit_name": "Software Engineering",
        "synopsis":"This unit provides an introduction to the discipline of software engineering at the postgraduate level.",
        "semester": 1,
        "year": [2016],
        "mark": 85
      },
      {
        "unit_code": "FIT5171",
        "unit_name": "System validation and verification, quality and standards",
        "synopsis":"This unit covers the core software engineering disciplines concerned with managing and delivering quality software.",
        "semester": 2,
        "year": [2016],
        "mark": 75
      }
    ]
}

{
    "sid": 127,
    "name": {
      "first": "Glenn",
      "last": "Adams"
    },
    "course": "MDS",
    "result": [
      {
        "unit_code": "FIT5202",
        "unit_name": "Data processing for big data",
        "synopsis":"This unit teaches about working with different kinds of data, documents, graphs, spatial data. Distributed processing is introduced using Hadoop and Spark technologies, including streaming, graph processing and using NoSQL.",
        "semester": 2,
        "year": [2017],
        "mark": 70
      }
    ]
}
```
  
</font><br>


Also, we will add a new field into the documents in the collection. The following is syntax for adding a new field:
```
db.FIT_COMPLEX.update({}, {$set: {"yearOfUni": 1}}, {multi:true})
```

Let's change the value of `yearOfUni`: 
<font color='blue'>
**Exercise**: set it to be 2 for documents with 'sid' = 123 and 'sid' = 124; set it to be 3 for documents with 'sid' = 125 and 'sid'=126; and set it to be 4 for documents with 'sid' = 127.
</font><br>


## Advanced Query Operators ##
Using advance query operators in MongoDB can help us to gain more flexible control over the documents in a MongoDB database. First we will look at comparison operators that can be used to conditionally retrieve documents.

Let's look at the usage of the following operators: `$eq, $lt, $lte, $gt, $gte, $in, $nin, and $not`. 

#### \$eq
First, we will learn how we can use $eq. This operator matches values that are equal to a specified value. 

The following queries documents whose `yearOfUni` is equal to 2:
```
db.FIT_COMPLEX.find ({yearOfUni: {$eq: 2}})
```

Simply, this is equivalent to the following:
```
db.FIT_COMPLEX.find ({yearOfUni: 2})
```

We can also use $eq to a field in embedded documents. For example, let's select all documents where the first name is equal to "Marie". To specify a condition on the a field in an embedded document, use the dot notation:

```
db.FIT_COMPLEX.find ({"name.first": {$eq: "Marie"}})
```

The query is also equivalent to:
```
db.FIT_COMPLEX.find ({"name.first": "Marie"})
```


#### $lt and $lte
If you want to find all students whose previous marks are less than 80, then execute the following:
```
db.FIT_COMPLEX.find({"result.mark":{"$lt":80}})
```

"$lte" can be used to match values that are "less than or equal to" a specified value.


#### \$gt and \$gte
\$gt and \$gte stand for 'greater than' and 'greater than or equal to' respectively. 

<font color='blue'>
**Exercise**: Find all students whose previous marks are greater than or equal to 90.
</font><br>

#### \$in and \$nin
If we want to find all students whose `yearOfUni` is either class 2 or 3, then run the following:
```
db.FIT_COMPLEX.find({"yearOfUni":{"$in":[2, 3]}})
```

Its inverse can be queried using \$nin. 
<font color='blue'>
**Exercise**: Let’s find students who are not 2 or 3 year of uni.
</font><br>

<font color='blue'>
**Exercise**: Let’s combine all of the above operators and write a query. Find all students that are either MIT or MDS students; these students must be either 3 or 4 year of unit; and also, at least the previous marks should be greater than or equal to 90. Note that at least you need to use "$in" and "$or" once for this query.
</font><br>

**Solution and Expected Output**: 
```
db.FIT_COMPLEX.find({"course":{"$in":["MIT", "MDS"]}, $or:[{"yearOfUni":2}, {"yearOfUni":3}], "result.mark":{"$gte":90}})
```

#### Pretty
We can use the `pretty()` method to look our result in a presentable manner. For example, run the following query:
```
db.FIT_COMPLEX.find({"course":/(MD|MI)/i}).pretty()
```
As can be seem, `pretty()` provides an easy-to-read output format.

## Regular Expressions ##
Now, we will learn how to use **regular expressions** in MongoDB. Regular expressions can be used for searching for a pattern of word in any string. MongoDB also provides functionality of regular expression for string pattern matching using the `$regex` operator. 

Let's see some examples. In the `FIT_COMPLEX` collection, if you want to find all students with their course name including “MD” or “MI”, then we can design a query like:
```
db.FIT_COMPLEX.find({"course":/(MD|MI)/i})
```

Let's understand the syntax of the above query:
- i: the regex is case insensitive (case sensitive: without 'i')
- (MD|MI): the course name string must include with either “MD” or “MI”.


Alternatively, using `$regex`, the above query can be written as:
```
db.FIT_COMPLEX.find({"course":{$regex: "(MD|MI)", $options:"$i"}})
```
As can be seen, to search for a string in the case insensitive manner, we need to use `$options` with its value as `$i`.

<font color='blue'>
**Exercise**: Let’s find students with their names including with "Da" or "Jo".
</font><br>
**Solution and Expected Output**: 
```
db.FIT_COMPLEX.find({"name.first":/(Da|Jo)/i})
```

Some other usefule expressions include:
- "course":/(^MD): all course names staring with 'MD'
- "course":/(MD$): all course names ending with 'MD'

<font color='blue'>
**Exercise**: Let’s find students with their course names staring with "MD". Also, let's find students with their course names ending with "DS". 
</font><br>
```
db.FIT_COMPLEX.find({"course":/(^MD)/})
db.FIT_COMPLEX.find({"course":/(DS$)/})
```

<font color='blue'>
**Exercise**: Next, let’s complicate the query a bit. Let’s combine it with the operators covered above.
Find students whose first names starting with 'A', 'J' or 'D'; their years of uni are greater than 2; and completed units with their names including 'Data'. You don't need to consider "case-sensitive" for this search.
</font><br>

**Solution and Expected Output**: 
```
db.FIT_COMPLEX.find({"name.first":/(^A|^J|^D)/i, "yearOfUni":{$gt:2}, "result.unit_name":/(Data)/i})

```

For more information about `$regex` please refer to the following:https://docs.mongodb.com/manual/reference/operator/query/regex/

## Text Search
Now let's focus on learning how the MongoDB text search engine is working. For this activity, we will continually use the `FIT_COMPLEX` collection.

Let's first initialise and check the current indexes. Run the following:
```
db.FIT_COMPLEX.dropIndexes()
db.FIT_COMPLEX.getIndexes()
```

Let's first create a text index on the embeded "synopsis" field. Run this query:
```
db.FIT_COMPLEX.createIndex({"result.synopsis":"text"})
```
Now MongoDB inserted this text index into our collection, and all future documents that have this field will be processed and searched through this index. You can check the indexes for the collection:
```
db.FIT_COMPLEX.getIndexes()
```

Okay, you can see now that a new index `result.synopsis` has been created.

Let’s run a text search command. Use `find()` to search for students that have completed units with their synopsis descriptions include a term, "data", on the `synopsis` field:
```
db.FIT_COMPLEX.find({$text:{$search:"data"}})
```

Check the result. To display the output documents in an easy-to-read format, we can `pretty()`.

<font color='blue'>
**Exercise**: Apply 'pretty()' on the above query.
</font><br>

Now we want to **sort the output documents by relevance**. Each output document is given a **relevance score** for use by the MongoDB engine. By sorting them, we can choose the best documents of the group. Let's get started! First, we need to get the relevance score for each document to the query:

````
db.FIT_COMPLEX.find({$text:{$search:"data"}}, {score:{$meta:"textScore"}}).pretty()
````

Here, we use `{score:{$meta:"textScore"}}` as an option to the query. Regarding the relevance score, the higher, the better the match.

Once we get the relevance score for each document, we can now sort the output documents by their relevance scores. We need to repeat `{score:{$meta:"textScore"}}` inside the `sort()` method as the previous one does not deliver this information to `sort()`.
````
db.FIT_COMPLEX.find({$text:{$search:"data"}}, {score:{$meta:"textScore"}}).pretty().sort({score:{$meta:"textScore"}})

````

Perhaps, we can see too long output to read. Let's limit the output so we can only see "sid", "result.synopsis", and relevance score:

````
db.FIT_COMPLEX.find({$text:{$search:"data"}}, {score:{$meta:"textScore"}, _id:0, sid:1, "result.synopsis":1}).pretty().sort({score:{$meta:"textScore"}})
````

Check the results with their relevance scores. Are these reasonable?

Of course, we can use regular expressions to search for a string. 

<font color='blue'>
**Exercise**: Using a regular expression, find students who have completed units with their synopsis has 'programming' (case insensitive).
</font><br>

**Solution and Expected Output**: 
````
db.FIT_COMPLEX.find({"result.synopsis":/programming/i})
````

## Aggregate

Aggregation operations process documents, group their values together, and perform a variety of operations on the grouped data to return a single result. Thus, aggregation provides a great way to massage data into the output you want. 

For this activity, let's insert two more documents into `FIT_COMPLEX` using `insert()`.

<font color='blue'>
**Exercise**: Insert the following 2 documents into 'FIT_COMPLEX'.
</font><br>

u5={
"sid": 128,
"name": {
  "first": "Berry",
  "last": "Ross"
},
"course": "MDS",
"result": [
  {
    "unit_code": "FIT5202",
    "unit_name": "Data processing for big data",
    "synopsis":"This unit teaches about working with different kinds of data, documents, graphs, spatial data. Distributed processing is introduced using Hadoop and Spark technologies, including streaming, graph processing and using NoSQL.",
    "semester": 2,
    "year": [2017],
    "mark": 80
  }
],
"yearOfUni": 3
}

u6={
"sid": 129,
"name": {
  "first": "David",
  "last": "Rod"
},
"course": "MDS",
"result": [
  {
    "unit_code": "FIT5202",
    "unit_name": "Data processing for big data",
    "synopsis":"This unit teaches about working with different kinds of data, documents, graphs, spatial data. Distributed processing is introduced using Hadoop and Spark technologies, including streaming, graph processing and using NoSQL.",
    "semester": 2,
    "year": [2017],
    "mark": 90
  }
],
"yearOfUni": 3
}

u8={
"sid": 130,
"name": {
  "first": "Kevin",
  "last": "McDonald"
},
"course": "MIT",
"result": [
  {
    "unit_code": "FIT5136",
    "unit_name": "Software Engineering",
    "synopsis":"This unit provides an introduction to the discipline of software engineering at the postgraduate level.",
    "semester": 1,
    "year": [2017],
    "mark": 90
  }
],
"yearOfUni": 3
}
  
  
Now, we can find the number of the students in each year of uni. using `aggregate()`:
````
db.FIT_COMPLEX.aggregate({$group:{_id:"$yearOfUni", count:{$sum:1}}})
````

<font color='blue'>
**Exercise**: Find the number of students in each course.
</font><br>

**Solution and Expected Output**: 
```
db.FIT_COMPLEX.aggregate({$group:{_id:"$course", count:{$sum:1}}})
```

Similarly, we can also find the "course-wise" average year of uni:
```
db.FIT_COMPLEX.aggregate({$group:{_id:"$course", avg:{$avg:"$yearOfUni"}}})
```

## Joining Two Collections 

How can we merge data from two collections? To address this issue, we can use the **"`$lookup`" operator in aggregation framework** which can be utilized to perfor **LEFT JOIN**.

JOIN is one of the key difference between SQL and NoSQL database. MongoDB Aggregation `$lookup` operator is useful to get JOIN for two collections like doing it in RDBMS. 

Let's first understand the syntax of `$lookup`:
```
db.collection.aggregation([
    {
        $lookup:{
            from: "[foreign collection]",
            localField : "[local field]",
            foreignField : <field from the documents of the "from" collection>
            as : "[key name to appear in result]"
        }
    }
])
```

- from:	the foreign collection in the same database to perform the join with.
- localField: the field of input documents. If an input document does not contain the localField, `$lookup` treats the field as 'null' for matching purposes.
- foreignField:	the field from the documents in the from collection. If a document in the from collection does not contain the foreignField, `$lookup` treats as 'null' for matching purposes.
- as: the name of the new array field to add to the input documents. The new array field contains the matching documents from the from collection. 

Let's make some examples of using `$lookup`. For this, we will make two new collections: (1) users and (2) units. 

<font color='blue'>
**Exercise**: First, let's create two users and add them into the `users` collection.
</font><br>

```
{
    "sid": 123,
    "name": {
      "first": "Marie",
      "last": "Currie"
    }
}

{
    "sid": 124,
    "name": {
      "first": "Albert",
      "last": "Einstein"
    }
}    
```

<font color='blue'>
**Exercise**: Second, let's create three units that these two users have completed. Insert them into the `units` collection:
</font><br>
```
{
    "sid": 123,
    "unit_code": "FIT9132",
    "unit_name": "Database",
    "semester": 1,
    "year": [2017],
    "mark": 100
}
      
{
    "sid": 123,
    "unit_code": "FIT9131",
    "unit_name": "Programming",
    "semester": [1,2],
    "year": [2016],
    "mark": 80
}

{
    "sid": 124,
    "unit_code": "FIT9132",
    "unit_name": "Database",
    "semester": 2,
    "year": [2017],
    "mark": 100
}
```

Now, let's retrieve users with their completed units as:

```
db.users.aggregate({
$lookup:
    {
        from: "units",
        localField: "sid",
        foreignField : "sid",
        as: "completed_units"
    }
}).pretty()
```

As you can see, a user may contain multiple units completed. Please check how you can see the results within the array (i.e. "completed_unit"). Can we create separate document per unit including a use who has completed? You we can do it using `$unwind`.

Let's run the following:
```
db.getCollection('units').aggregate([
   {
      $unwind: "$sid"
   },
   {
    $lookup:
        {
            from: "users",
            localField: "sid",
            foreignField : "sid",
            as: "completed sid"
        }
    }
]).pretty()
```

To sum up, `$unwind` can be used to peel off the elements of an array individually, and returns a stream of documents.
That is `$unwind` works on the the array field inside the documents, and creates a new document for each array element in an array. So its output is a new document of each entry of an array inside a document. Thus, we can use `$unwind` to flattens the data.


## Modeling Data Schema

MongoDB doesn't impose a particular schema on data being input. However, it is still necessary to understand and build good data schemas (or models) that we use in a database. Also, modeling the data makes your queries as efficient as possible. Thus, give data, how to model the data is very imporant when designing your collections and databases.

Here, we are working on two collections: `users` and `units`. But we will modify these collections a bit to demonstrate to explain two different data models.


<font color='blue'>
**Exercise**: Let's first drop the two collections we created in the previous section: 'users' and 'units'.
</font><br>

Let's create two new collections: `users` and `units`. First, we will create two users and insert them into the `users` collection:
```
{
    "sid": 123,
    "name": {
      "first": "Marie",
      "last": "Currie"
    },
   "completed_units": ["FIT9131", "FIT9132"]
}

{
    "sid": 124,
    "name": {
      "first": "Albert",
      "last": "Einstein"
    },
   "completed_units": ["FIT9132"]
}    
```

Now, let's create these two units and insert them into the `units` collection:

```
{
    "unit_code": "FIT9131",
    "unit_name": "Programming",
    "synopsis":"This unit aims to provide students with the basic concepts involved in the development of well structured software using a programming language."
} 
 
{ 
    "unit_code": "FIT9132",
    "unit_name": "Database",
    "synopsis":"This unit will introduce the concept of data management in an organisation through relational database technology."
}
```

As can be seen, each user has maintained the units completed. And each unit has only information about its code, name, and synopsis. **Through these collections, we will think about how we can build different data schemas.**

We will discuss about two ways of modeling schemas: 
- **embedding (inclusion into another document)**, and 
- **referencing (reference documents in another collection)**.

We have an array of students in the `users` collection, and those unit codes completed will map to items (i.e. units) in the `units` collection. So basically, for any query involving both a student and a related unit, we need to consider both collections for the query.

<font color='blue'>
**Exercise**: Let's do a simple query. Find a student with 'sid = 123' in the 'users' collection. Use `pretty()` to display the result in a more formatted way.
</font><br>

**Solution and Expected Output**: 
```
db.users.find({"sid":123}).pretty()
```

<font color='blue'>
**Exercise**: Let's do another query in the 'units' collection. Find the 'FIT9131' unit and use `pretty()` as above
</font><br>

**Solution and Expected Output**: 
```
db.units.find({"unit_code":"FIT9131"}).pretty()
```

As can be seen, the output unit doesn't include who completed it. So we need to figure that out. How could we figure out which students have completed a particular unit? Remember that a student has an array of units that he/she completed, and Mongo can do a great job with that information.

<font color='blue'>
**Exercise**: Let's do a query about finding students who completed 'FIT9132'.
</font><br>

**Solution and Expected Output**: 
```
db.users.find({"completed_units":"FIT9132"}).pretty()
```

What's the result? Yes we got two students: 'sid=123' and 'sid=124'. How can we combine information from different collections together? To answer this question, we can build a embedding data model by writing some code.

We will take through the following steps and write some code on the `mongo` shell:
- Pick up a student from the `users` collection, 
- Get the information about that student, and 
- Combine it with a list of units from the `units` collection. 

Try with the following code:

```
var unitCode = "FIT9132"
var unitObj = db.units.findOne({"unit_code":unitCode}) // find the unit
unitObj.studentList = [] // create an emptry array
var studentList = db.users.find({"completed_units":unitCode}) // find students having the unitCode
studentList.forEach(function(student) { // for each student in the studentList
... unitObj.studentList.push(student) // push that student into the unitobj.studentList array
... })
> unitObj
```

Let's analyse the above code. First, we have the unit code, `FIT9132`. Then, we find the unit document having this unit code. There are two students who completed this unit, `sid=123` and `sid=124`. Then, we combine the information together using a for loop.

Let's summarise. MongoDB enables us to use two options for designing a data model: `embedding` and `referencing`. In our exercise, in the embedding design, we stored the student information in a unit document as we did in the code above. On the other hand, in the referencing design, you could normalise the model a bit by referencing the unit information in the `users` collection (i.e. our original data model: `users` and '`units`).

What's the benefit of using the embedding approach? We can store and retrieve all the related data a single seek. However, in some cases, a referencing model may work better - one benefit is that we can increase flexibility when querying the data. 

**Congratulations on finishing this activity!**

Next week, we will learn how we use Python to build an MongoDB application! See you next week!